- run tests on GPU / CPU
- Maybe also f32 / f64

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)

In [ ]:
import timeit
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import jax.numpy as jnp
import equinox as eqx

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.model_interfaces.model_interface import ModelInterface

from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import (
    FINAL_MATERIALS,
    TestSet,
    ResultSet,
    predict_test_scenarios,
    validate_result_set,
    visualize_result_set,
    evaluate_test_scenarios,
    update_pareto_df,
    get_exp_ids_per_material,
    predict_test_scenarios_single_material
)
from rhmag.model_setup import setup_normalizer, setup_dataset

In [ ]:
model_type = "GRU8"

exp_ids = get_exp_ids(
    model_type=model_type,
    exp_name="final-reduced-features-f32",
    enforce_identical_exp_name=True,
)
exp_ids

In [ ]:
def generate_dummy_data(key, past_len, future_len, batch_size):
    key, subkey = jax.random.split(key, 2)
    subkeys = jax.random.split(subkey, 4)
    
    B_past = jax.random.normal(subkeys[0], shape=(batch_size, past_len))
    H_past = jax.random.normal(subkeys[2], shape=(batch_size, past_len))
    B_future = jax.random.normal(subkeys[1], shape=(batch_size, future_len))
    T = jax.random.normal(subkeys[3], shape=(batch_size,))

    return B_past, H_past, B_future, T, key

In [ ]:
import matplotlib as mpl
from matplotlib import rc
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter, LogLocator
import matplotlib.ticker as ticker

rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

fullwidth = 7.167
columnwidth = 3.5

## batch size 100 - 900

In [ ]:
df_stored = pd.read_pickle("wall_time_test_results.pkl")

key = jax.random.key(2)

past_len = 100
future_len = 900

wall_times = []

gpus = jax.devices()

exp_ids = [
    'B_GRU32_pareto-front-f32_07dfa507_seed899',
    'D_GRU8_final-reduced-features-f32_3d0f8de4_seed12',
    'E_GRU64_pareto-front-f32_0b56149b_seed202',
]

for max_batch_size, device in zip([3.5, 5.5], ["cpu", gpus[-1]]):
    with jax.default_device(device):

        for exp_id in exp_ids:
            model_type = exp_id.split("_")[1]
            model = reconstruct_model_from_file(exp_id)
            n_params = model.n_params
            model = eqx.filter_jit(model)

            if model_type == "GRU8":
                if device == "cpu":
                    max_batch_size = 4.5
            
            if model_type == "GRU64":
                if device == "cpu":
                    max_batch_size = 3
                else:
                    max_batch_size = 5.5
            
            for batch_size in jnp.logspace(1, max_batch_size, 20, dtype=jnp.int32):

                # check if the entry already exists in the df
                exists = ((df_stored['model_type'] == model_type) & (df_stored['device'] == str(device)) & (df_stored['batch_size'] == batch_size)).any()
                if exists:
                    continue

                print(batch_size)
                
                B_past, H_past, B_future, T, key = generate_dummy_data(key, past_len, future_len, int(batch_size))
        
                # run once to ensure that any compilation is done before timing
                model(B_past, H_past, B_future, T).block_until_ready()
        
                t = timeit.Timer(lambda: model(B_past, H_past, B_future, T).block_until_ready())
                wall_time = t.repeat(repeat=100, number=1)
               
                wall_times.append(
                    {
                        "exp_id": exp_id,
                        "model_type": model_type,
                        "batch_size": batch_size,
                        "n_params": n_params,
                        "past_len": past_len,
                        "future_len": future_len,
                        "device": str(device),
                        "wall_time": wall_time,
                    }
                )

df_new = pd.DataFrame(wall_times)
df_new["wt_min"]  = df_new["wall_time"].apply(np.min)
df_new["wt_max"]  = df_new["wall_time"].apply(np.max)
df_new["wt_mean"] = df_new["wall_time"].apply(np.mean)
df_new["wt_std"]  = df_new["wall_time"].apply(np.std)

df = pd.concat([df_stored, df_new], ignore_index=True)
df.to_pickle("wall_time_test_results.pkl")

In [ ]:
# df_new = pd.DataFrame(wall_times)
# df_new["wt_min"]  = df_new["wall_time"].apply(np.min)
# df_new["wt_max"]  = df_new["wall_time"].apply(np.max)
# df_new["wt_mean"] = df_new["wall_time"].apply(np.mean)
# df_new["wt_std"]  = df_new["wall_time"].apply(np.std)

# df = pd.concat([df_stored, df_new], ignore_index=True)

# df.to_pickle("wall_time_test_results.pkl")

In [ ]:
df

In [ ]:
# fig, axs = plt.subplots(1, 1, figsize=(columnwidth, columnwidth/2))

# label_map = {
#     "cpu": "CPU",
#     "cuda:2": "GPU",
# }

# df_seq_len = df.loc[df['future_len'] == 900]

# ax = axs
# for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df_seq_len.groupby("device")):
   
#     batch_size = np.asarray(df_device["batch_size"], dtype=float)
#     mean = np.asarray(df_device["wt_mean"], dtype=float)
#     std = np.asarray(df_device["wt_std"], dtype=float)

#     ax.plot(batch_size, mean, color=color, label=label_map[device])
#     ax.fill_between(
#         batch_size,
#         mean + std,
#         mean - std,
#         color=color,
#         alpha=0.3,
#     )

# ax.set_xscale("log")
# ax.tick_params(which="both", axis="y", direction="in")
# ax.tick_params(which="both", axis="x", direction="in")
# ax.xaxis.minorticks_off()
# ax.grid(True, which="both", alpha=0.3)
    
# ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
# ax.set_xlabel("batch size $b$")
# ax.legend()

# ax.set_xlim(batch_size[0], batch_size[-1])
# plt.show()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(columnwidth, columnwidth), constrained_layout=True)

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

linestyle_map = {
    "GRU8": "-",
    "GRU32": "--",
    "GRU64": ":",
}

colors = ["tab:blue", "tab:orange"]
# colors = ["k", "tab:green"]

df_seq_len = df.loc[df['future_len'] == 900]

ax = axs[0]
for color, (device, df_device) in zip(colors, df_seq_len.groupby("device")):
    for idx, (model_type, df_model_type) in enumerate(df_device.groupby("model_type")):

        df_model_type = df_model_type.sort_values(by=["batch_size"], ascending=True)
        
        batch_size = np.asarray(df_model_type["batch_size"], dtype=float)
        mean = np.asarray(df_model_type["wt_mean"], dtype=float)
        std = np.asarray(df_model_type["wt_std"], dtype=float)
    
        ax.plot(batch_size, mean, color=color, label=label_map[device] + " -- " + model_type, linestyle=linestyle_map[model_type])
        ax.fill_between(
            batch_size,
            mean + std,
            mean - std,
            color=color,
            alpha=0.3,
        )

# ax.set_xscale("log")
# ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")

def formatter(val, pos):
    exp = 5
    mantissa = val / 10**exp
    # if mantissa == 1:
    #     return r'$\vphantom{2 \times} 10^{%d}$' % exp
    return r'$%d \times 10^{%d}$' % (mantissa, exp)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(formatter))


ax.xaxis.minorticks_off()
ax.yaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_ylim(3e-3, 1.5)

ax = axs[1]
for color, (device, df_device) in zip(colors, df_seq_len.groupby("device")):
    for idx, (model_type, df_model_type) in enumerate(df_device.groupby("model_type")):

        df_model_type = df_model_type.sort_values(by=["batch_size"], ascending=True)
        
        batch_size = np.asarray(df_model_type["batch_size"], dtype=float)
        mean = np.asarray(df_model_type["wt_mean"], dtype=float)
        std = np.asarray(df_model_type["wt_std"], dtype=float)
    
        ax.plot(batch_size, mean, color=color, label=label_map[device] + " -- " + model_type, linestyle=linestyle_map[model_type])
        ax.fill_between(
            batch_size,
            mean + std,
            mean - std,
            color=color,
            alpha=0.3,
        )

ax.set_xscale("log")
ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.yaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.2f}'))
ax.yaxis.set_minor_formatter(StrMethodFormatter('{x:.2f}'))

ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("batch size $b$")


handles, labels = ax.get_legend_handles_labels()  # fig.gca().get_legend_handles_labels()

print(handles)


order = [2, 0, 1, 5, 3, 4]
ax.legend(
   [handles[i] for i in order], [labels[i] for i in order], loc='upper center', bbox_to_anchor=(0.4, -0.35), ncol=2
)

plt.gcf().align_ylabels()

ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_ylim(3e-3, 0.25)
plt.savefig("wall_time_comparison_batch_size_model_size.pdf", bbox_inches="tight")
plt.show()

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(df)

maybe describe starting points for CPU and GPU numerically, then give an intuition when it starts to rise strongly.

So with the GPU, one could easily run $10^4$ models in parallel.

In [ ]:
raise

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(columnwidth, columnwidth/2))

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

linestyle_map = {
    "GRU8": "-",
    "GRU32": "--",
    "GRU64": ":",
}

df_seq_len = df.loc[df['future_len'] == 900]

ax = axs
for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df_seq_len.groupby("device")):
    for idx, (model_type, df_model_type) in enumerate(df_device.groupby("model_type")):

        df_model_type = df_model_type.sort_values(by=["batch_size"], ascending=True)
        
        batch_size = np.asarray(df_model_type["batch_size"], dtype=float)
        mean = np.asarray(df_model_type["wt_mean"], dtype=float)
        std = np.asarray(df_model_type["wt_std"], dtype=float)
    
        ax.plot(batch_size, mean, color=color, label=label_map[device] + " -- " + model_type, linestyle=linestyle_map[model_type])
        ax.fill_between(
            batch_size,
            mean + std,
            mean - std,
            color=color,
            alpha=0.3,
        )

ax.set_xscale("log")
ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.yaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("batch size $b$")

handles, labels = ax.get_legend_handles_labels()  # fig.gca().get_legend_handles_labels()

print(handles)


order = [2, 0, 1, 5, 3, 4]
ax.legend(
   [handles[i] for i in order], [labels[i] for i in order], loc='upper center', bbox_to_anchor=(0.4, -0.25), ncol=2
)

ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_ylim(3e-3, 0.25)
plt.savefig("wall_time_comparison_batch_size_model_size.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(columnwidth, columnwidth/2))

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

linestyle_map = {
    "GRU8": "-",
    "GRU32": "--",
    "GRU64": ":",
}

df_seq_len = df.loc[df['future_len'] == 900]

ax = axs
for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df_seq_len.groupby("device")):
    for idx, (model_type, df_model_type) in enumerate(df_device.groupby("model_type")):

        df_model_type = df_model_type.sort_values(by=["batch_size"], ascending=True)
        
        batch_size = np.asarray(df_model_type["batch_size"], dtype=float)
        mean = np.asarray(df_model_type["wt_mean"], dtype=float)
        std = np.asarray(df_model_type["wt_std"], dtype=float)
    
        ax.plot(batch_size, mean, color=color, label=label_map[device] + " -- " + model_type, linestyle=linestyle_map[model_type],)# s=4)
        ax.fill_between(
            batch_size,
            mean + std,
            mean - std,
            color=color,
            alpha=0.3,
        )

ax.set_xscale("log")
#ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.yaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("batch size $b$")

handles, labels = ax.get_legend_handles_labels()  # fig.gca().get_legend_handles_labels()

print(handles)


order = [2, 0, 1, 5, 3, 4]
ax.legend(
   [handles[i] for i in order], [labels[i] for i in order], loc='upper center', bbox_to_anchor=(0.4, -0.25), ncol=2
)

ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_ylim(3e-3, 0.25)
plt.savefig("wall_time_comparison_batch_size_model_size.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Different model sizes
# different sequence lengths?

# Not sure if this should go into the paper? But still probably makes sense to check it out
# ensure that there is a roughly proportional scale between # params and computation time?

## future len scale

In [ ]:
key = jax.random.key(2)

past_len = 1

wall_times = []

gpus = jax.devices()

exp_ids = [
    'D_GRU8_final-reduced-features-f32_3d0f8de4_seed12'
]

for max_batch_size, device in zip([3, 3], ["cpu", gpus[-1]]):
    with jax.default_device(device):
        for exp_id in exp_ids:
            model = reconstruct_model_from_file(exp_id)
            n_params = model.n_params
            model = eqx.filter_jit(model)

            for future_len in [1, 900, 5000, 10_000, 20_000, 50_000, 100_000]:
                batch_size = 1
                B_past, H_past, B_future, T, key = generate_dummy_data(key, past_len, future_len, int(batch_size))
        
                # run once to ensure that any compilation is done before timing
                model(B_past, H_past, B_future, T).block_until_ready()
        
                t = timeit.Timer(lambda: model(B_past, H_past, B_future, T).block_until_ready())
                wall_time = t.repeat(repeat=20, number=1)
               
                wall_times.append(
                    {
                        "exp_id": exp_id,
                        "model_type": model_type,
                        "batch_size": batch_size,
                        "n_params": n_params,
                        "past_len": past_len,
                        "future_len": future_len,
                        "device": str(device),
                        "wall_time": wall_time,
                    }
                )

df = pd.DataFrame(wall_times)
df["wt_min"]  = df["wall_time"].apply(np.min)
df["wt_max"]  = df["wall_time"].apply(np.max)
df["wt_mean"] = df["wall_time"].apply(np.mean)
df["wt_std"]  = df["wall_time"].apply(np.std)

In [ ]:
# I expect a linear increase in computation time. Do we see that?

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(columnwidth, columnwidth/2))

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

ax = axs
for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df.groupby("device")):
   
    batch_size = np.asarray(df_device["future_len"], dtype=float)
    mean = np.asarray(df_device["wt_mean"], dtype=float)
    std = np.asarray(df_device["wt_std"], dtype=float)

    ax.plot(batch_size, mean, color=color, label=label_map[device])
    ax.fill_between(
        batch_size,
        mean + std,
        mean - std,
        color=color,
        alpha=0.3,
    )

# ax.set_xscale("log")
# ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("sequence length $l$")
ax.legend()

ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_ylim(1e-5, 0.1)
plt.show()

In [ ]:
raise

## batch size 1 - 1

In [ ]:
get_exp_ids(
    material_name="E",
    model_type="GRU64",
    exp_name="pareto",
    enforce_identical_exp_name=False
)

In [ ]:
reconstruct_model_from_file().n_params

In [ ]:
key = jax.random.key(2)

past_len = 1
future_len = 1

wall_times = []

gpus = jax.devices()
cpu_tag = "cpu"

exp_ids = [
    'B_GRU32_pareto-front-f32_07dfa507_seed899',
    'D_GRU8_final-reduced-features-f32_3d0f8de4_seed12',
    'E_GRU64_pareto-front-f32_0b56149b_seed202',
]

# for max_batch_size, device in zip([7, 7], ["cpu", gpus[-1]]):
for max_batch_size, device in zip([7], [gpus[-1]]):
    
    with jax.default_device(device):
        for exp_id in exp_ids:
            model_type = exp_id.split("_")[1]
            model = reconstruct_model_from_file(exp_id)
            n_params = model.n_params
            model = eqx.filter_jit(model)
            for batch_size in jnp.hstack([1, jnp.logspace(1, max_batch_size, 30, dtype=jnp.int32)]):
                print(batch_size)
                
                B_past, H_past, B_future, T, key = generate_dummy_data(key, past_len, future_len, int(batch_size))
        
                # run once to ensure that any compilation is done before timing
                model(B_past, H_past, B_future, T).block_until_ready()
        
                t = timeit.Timer(lambda: model(B_past, H_past, B_future, T).block_until_ready())
                wall_time = t.repeat(repeat=500, number=1)
               
                wall_times.append(
                    {
                        "exp_id": exp_id,
                        "model_type": model_type,
                        "batch_size": batch_size,
                        "n_params": n_params,
                        "past_len": past_len,
                        "future_len": future_len,
                        "device": str(device),
                        "wall_time": wall_time,
                    }
                )

df = pd.DataFrame(wall_times)
df["wt_min"]  = df["wall_time"].apply(np.min)
df["wt_max"]  = df["wall_time"].apply(np.max)
df["wt_mean"] = df["wall_time"].apply(np.mean)
df["wt_std"]  = df["wall_time"].apply(np.std)

In [ ]:
df = pd.DataFrame(wall_times)
df["wt_min"]  = df["wall_time"].apply(np.min)
df["wt_max"]  = df["wall_time"].apply(np.max)
df["wt_mean"] = df["wall_time"].apply(np.mean)
df["wt_std"]  = df["wall_time"].apply(np.std)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(12,12))#figsize=(columnwidth, columnwidth/2))

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

linestyle_map = {
    "GRU8": "-",
    "GRU32": "--",
    "GRU64": ":",
}

ax = axs
for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df.groupby("device")):

    for idx, (model_type, df_model_type) in enumerate(df_device.groupby("model_type")):
        print(model_type)
        batch_size = np.asarray(df_model_type["batch_size"], dtype=float)
        mean = np.asarray(df_model_type["wt_mean"], dtype=float)
        std = np.asarray(df_model_type["wt_std"], dtype=float)
    
        ax.plot(
            batch_size,
            mean,
            color=color,
            label=label_map[device] if idx == 0 else None,
            linestyle=linestyle_map[model_type],
        )
        ax.fill_between(
            batch_size,
            mean + std,
            mean - std,
            color=color,
            alpha=0.3,
        )

#ax.set_xscale("log")
ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.yaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("batch size $b$")
ax.legend()

ax.set_xlim(batch_size[0], batch_size[-1])
ax.set_xlim(1e4, 2 * 1e5)
#ax.set_ylim(None,)

plt.savefig("wall_time_comparison.pdf", bbox_inches="tight")
plt.show()


In [ ]:
df

How much memory is required?

## Model size influence:

In [ ]:
from rhmag.model_setup import setup_model, setup_featurize
from rhmag.data_management import Normalizer

In [ ]:
model, _ = setup_model(
    model_label="GRU32",
    model_key=jax.random.key(0),
    normalizer=Normalizer(
        B_max=0.5,
        H_max=200.1,
        T_max=70.0,
        norm_fe_max=[1.5, 5.62],
        H_transform=lambda x:x,
        H_inverse_transform=lambda x:x,
    ),
    featurize=setup_featurize(
        disable_features="reduce",
        dyn_avg_kernel_size=None,
        time_shift=None
    )
)
model.n_params

In [ ]:
key = jax.random.key(1)

past_len = 100
future_len = 900

wall_times = []

gpus = jax.devices()

device = gpus[-1]

batch_size = 10

# for max_batch_size, device in zip([3, 5.5], ["cpu", gpus[-1]]):
with jax.default_device(device):
    for n_hidden_units in [2, 5, 7, 10, 15, 17, 25, 30, 32, 40, 50, 70, 90, 125, 170, 200, 300, 400, 500, 700, 1000, 1250, 1500, 2000]:
        print(n_hidden_units)
        model, _ = setup_model(
            model_label=f"GRU{int(n_hidden_units)}",
            model_key=jax.random.key(0),
            normalizer=Normalizer(
                B_max=0.5,
                H_max=200.1,
                T_max=70.0,
                norm_fe_max=[1.5, 5.62],
                H_transform=lambda x:x,
                H_inverse_transform=lambda x:x,
            ),
            featurize=setup_featurize(
                disable_features="reduce",
                dyn_avg_kernel_size=None,
                time_shift=None
            )
        )
        n_params = model.n_params
        model = eqx.filter_jit(model)                
        B_past, H_past, B_future, T, key = generate_dummy_data(key, past_len, future_len, int(batch_size))

        # run once to ensure that any compilation is done before timing
        model(B_past, H_past, B_future, T).block_until_ready()

        t = timeit.Timer(lambda: model(B_past, H_past, B_future, T).block_until_ready())
        wall_time = t.repeat(repeat=50, number=1)
       
        wall_times.append(
            {
                "exp_id": None,
                "model_type": model_type,
                "batch_size": batch_size,
                "n_params": n_params,
                "past_len": past_len,
                "future_len": future_len,
                "device": str(device),
                "wall_time": wall_time,
            }
        )

df = pd.DataFrame(wall_times)
df["wt_min"]  = df["wall_time"].apply(np.min)
df["wt_max"]  = df["wall_time"].apply(np.max)
df["wt_mean"] = df["wall_time"].apply(np.mean)
df["wt_std"]  = df["wall_time"].apply(np.std)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(columnwidth, columnwidth/2))

label_map = {
    "cpu": "CPU",
    "cuda:2": "GPU",
}

df_seq_len = df.loc[df['future_len'] == 900]

ax = axs
for color, (device, df_device) in zip(["tab:blue", "tab:orange"], df_seq_len.groupby("device")):
   
    n_params = np.asarray(df_device["n_params"], dtype=float)
    mean = np.asarray(df_device["wt_mean"], dtype=float)
    std = np.asarray(df_device["wt_std"], dtype=float)

    ax.plot(n_params, mean, color=color, label=label_map[device])
    ax.fill_between(
        n_params,
        mean + std,
        mean - std,
        color=color,
        alpha=0.3,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.tick_params(which="both", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")
ax.xaxis.minorticks_off()
ax.grid(True, which="both", alpha=0.3)
    
ax.set_ylabel(r"$t_{\mathrm{w}}$ in s")
ax.set_xlabel("\\# params")
ax.legend()

ax.set_xlim(n_params[0], n_params[-1])
plt.show()

In [ ]:
df